# 面试问题：N-BEATS 怎样用 Backcast 残差堆叠多个 Block 做多步时间序列预测？

## 可直接复述的回答主线

1. N-BEATS 每个 Block 从历史窗口同时输出 backcast 和 forecast。
2. 下一 Block 接收 residual=x-backcast，而不是再次接收原始 x；多个 forecast 相加形成最终预测。
3. Generic Block 可以用 MLP 直接生成 backcast/forecast，不依赖 RNN 或一站式时序框架。
4. 训练时按时间顺序切窗口并只用过去统计量归一化，避免未来数据泄漏。
5. 评测应逐 origin 输出多步预测、MAE、Block backcast、residual 和 forecast 分解。
6. 生产还需缺失值、节假日外生变量、多序列共享、滚动回测、概率预测、漂移和预测区间。

后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是 56 天脱敏订单量，包含线性增长和固定周内季节性。前段生成训练窗口，最后 6 个 forecast origin 做滚动测试；每个输入 14 天、预测未来 3 天。确定性序列只用于解释 Block 残差机制。

In [1]:
import math  # 生成周期订单并计算 MAE。
import torch  # 使用基础 PyTorch MLP 和自动微分实现 N-BEATS。
torch.manual_seed(105)  # 固定网络初始化和训练轨迹。
weekly_pattern = [12.0, 5.0, -2.0, -6.0, -1.0, 8.0, 16.0]  # 定义周一到周日的业务季节项。
deterministic_noise = [0.0, 1.0, -1.0, 0.5, -0.5, 1.5, -1.5]  # 定义每周重复的小幅未建模扰动。
daily_orders = [80.0 + 0.75 * day + weekly_pattern[day % 7] + deterministic_noise[(day * 3) % 7] for day in range(56)]  # 生成五十六天趋势加周季节订单量。
dates = [f"day-{day + 1:02d}" for day in range(56)]  # 为每个日订单生成可读日期 ID。
input_length = 14  # 设定历史输入窗口为两周。
horizon = 3  # 设定多步预测未来三天。
test_origins = list(range(46, 52))  # 选择最后六个具有完整三日目标的滚动 origin。
training_origins = list(range(input_length, 44))  # 只用早于测试区间的三十个训练 origin。
def window_tensors(origins):  # 把时间 origin 转为输入和未来目标张量。
    inputs = torch.tensor([daily_orders[origin - input_length:origin] for origin in origins], dtype=torch.float32)  # 构造样本乘十四历史矩阵。
    targets = torch.tensor([daily_orders[origin:origin + horizon] for origin in origins], dtype=torch.float32)  # 构造样本乘三未来目标。
    return inputs, targets  # 返回监督时序窗口。
training_inputs_raw, training_targets_raw = window_tensors(training_origins)  # 构造训练窗口。
test_inputs_raw, test_targets_raw = window_tensors(test_origins)  # 构造六个滚动测试窗口。
normalization_mean = training_inputs_raw.mean()  # 仅用训练历史计算全局均值。
normalization_std = training_inputs_raw.std().clamp_min(1.0e-6)  # 仅用训练历史计算标准差。
training_inputs = (training_inputs_raw - normalization_mean) / normalization_std  # 标准化训练历史。
training_targets = (training_targets_raw - normalization_mean) / normalization_std  # 用同一训练统计量标准化目标。
test_inputs = (test_inputs_raw - normalization_mean) / normalization_std  # 标准化测试输入但不重估统计量。
print("教学实验输入：56天订单量")  # 标记下方为确定性业务时序。
print("前14天=", list(zip(dates[:14], [round(value, 2) for value in daily_orders[:14]])))  # 展示两周趋势与季节性。
print("后14天=", list(zip(dates[-14:], [round(value, 2) for value in daily_orders[-14:]])))  # 展示测试附近真实订单。
print("train/test window shapes=", tuple(training_inputs.shape), tuple(test_inputs.shape), "mean/std=", normalization_mean.item(), normalization_std.item())  # 展示数据切分和无泄漏归一化。

教学实验输入：56天订单量
前14天= [('day-01', 92.0), ('day-02', 86.25), ('day-03', 78.0), ('day-04', 75.25), ('day-05', 83.5), ('day-06', 92.75), ('day-07', 100.0), ('day-08', 97.25), ('day-09', 91.5), ('day-10', 83.25), ('day-11', 80.5), ('day-12', 88.75), ('day-13', 98.0), ('day-14', 105.25)]
后14天= [('day-43', 123.5), ('day-44', 117.75), ('day-45', 109.5), ('day-46', 106.75), ('day-47', 115.0), ('day-48', 124.25), ('day-49', 131.5), ('day-50', 128.75), ('day-51', 123.0), ('day-52', 114.75), ('day-53', 112.0), ('day-54', 120.25), ('day-55', 129.5), ('day-56', 136.75)]
train/test window shapes= (30, 14) (6, 14) mean/std= 100.32142639160156 10.530779838562012


## 2. Baseline / 基线：最后一个观测值重复三步

Naive forecast 假设未来三天都等于 origin 前一天。它没有学习周内周期，周末到工作日切换时误差明显。

In [2]:
baseline_predictions = test_inputs_raw[:, -1:].repeat(1, horizon)  # 把每个测试窗口最后订单量复制三步。
baseline_absolute_errors = torch.abs(baseline_predictions - test_targets_raw)  # 计算逐 origin、逐 horizon 绝对误差。
baseline_mae = float(baseline_absolute_errors.mean().item())  # 汇总 Naive 多步 MAE。
print("Baseline last-value 多步预测")  # 标记下表展示周期切换失败。
print("origin   last_value  target未来3天              prediction                MAE")  # 输出基线结果表头。
for origin, prediction, target, errors in zip(test_origins, baseline_predictions, test_targets_raw, baseline_absolute_errors):  # 逐滚动 origin 展示预测。
    print(f"{dates[origin]:<8} {daily_orders[origin - 1]:>10.2f} {target.tolist()!s:<26} {prediction.tolist()!s:<26} {errors.mean().item():.3f}")  # 输出当前三步目标和误差。
print(f"Baseline MAE={baseline_mae:.4f}")  # 展示同数据基线误差。

Baseline last-value 多步预测
origin   last_value  target未来3天              prediction                MAE
day-47       106.75 [115.0, 124.25, 131.5]     [106.75, 106.75, 106.75]   16.833
day-48       115.00 [124.25, 131.5, 128.75]    [115.0, 115.0, 115.0]      13.167
day-49       124.25 [131.5, 128.75, 123.0]     [124.25, 124.25, 124.25]   4.333
day-50       131.50 [128.75, 123.0, 114.75]    [131.5, 131.5, 131.5]      9.333
day-51       128.75 [123.0, 114.75, 112.0]     [128.75, 128.75, 128.75]   12.167
day-52       123.00 [114.75, 112.0, 120.25]    [123.0, 123.0, 123.0]      7.333
Baseline MAE=10.5278


## 3. 底层实现：Generic Block、Backcast Residual 与 Forecast 累加

每个 Block 用两层 MLP 编码 14 天输入，再分别输出 14 维 backcast 和 3 维 forecast。第二 Block 显式接收 `x-backcast1`。

In [3]:
class NBeatsBlock(torch.nn.Module):  # 定义显式 forward 的 Generic N-BEATS Block。
    def __init__(self, input_length, horizon, hidden_dim=64):  # 初始化共享 MLP 和两个输出头。
        super().__init__()  # 注册 PyTorch 参数管理。
        self.hidden = torch.nn.Sequential(torch.nn.Linear(input_length, hidden_dim), torch.nn.ReLU(), torch.nn.Linear(hidden_dim, hidden_dim), torch.nn.ReLU())  # 编码历史窗口。
        self.backcast_head = torch.nn.Linear(hidden_dim, input_length)  # 输出要从输入解释掉的历史分量。
        self.forecast_head = torch.nn.Linear(hidden_dim, horizon)  # 输出当前 Block 的未来贡献。
    def forward(self, inputs):  # 对一批历史窗口同时生成 backcast 和 forecast。
        hidden = self.hidden(inputs)  # 提取当前 Block 隐藏表示。
        backcast = self.backcast_head(hidden)  # 生成十四维历史解释。
        forecast = self.forecast_head(hidden)  # 生成三维未来预测贡献。
        return backcast, forecast  # 返回供残差堆叠的两种输出。
class NBeats(torch.nn.Module):  # 定义两个 Generic Block 的显式堆叠。
    def __init__(self, input_length, horizon):  # 初始化两个独立 Block。
        super().__init__()  # 注册子模块参数。
        self.block_one = NBeatsBlock(input_length, horizon)  # 创建第一层趋势/季节解释 Block。
        self.block_two = NBeatsBlock(input_length, horizon)  # 创建处理剩余误差的第二 Block。
    def forward(self, inputs, return_debug=False):  # 执行 backcast 残差和 forecast 累加。
        backcast_one, forecast_one = self.block_one(inputs)  # 第一 Block 解释原始历史。
        residual_one = inputs - backcast_one  # 从输入减去第一 backcast 得到残差。
        backcast_two, forecast_two = self.block_two(residual_one)  # 第二 Block 只处理剩余历史。
        residual_two = residual_one - backcast_two  # 计算最终未解释残差。
        forecast = forecast_one + forecast_two  # 累加两个 Block 的未来贡献。
        if return_debug:  # 检查调用方是否需要分解中间量。
            return forecast, {"backcast_one": backcast_one, "residual_one": residual_one, "forecast_one": forecast_one, "backcast_two": backcast_two, "residual_two": residual_two, "forecast_two": forecast_two}  # 返回预测和 Block 分解。
        return forecast  # 返回最终三步预测。
model = NBeats(input_length, horizon)  # 创建两 Block N-BEATS。
optimizer = torch.optim.Adam(model.parameters(), lr=0.008)  # 创建参数更新器。
history = []  # 保存真实 backward 训练轨迹。
for step in range(700):  # 对三十个训练窗口执行全批次训练。
    optimizer.zero_grad(set_to_none=True)  # 清除上一步梯度。
    normalized_forecast, debug = model(training_inputs, return_debug=True)  # 前向执行 backcast residual 堆叠。
    forecast_loss = ((normalized_forecast - training_targets) ** 2).mean()  # 计算多步 forecast MSE。
    residual_penalty = 0.02 * (debug["residual_two"] ** 2).mean()  # 轻度约束最终历史残差避免两个 Block 漂移。
    loss = forecast_loss + residual_penalty  # 合并预测与 backcast 残差目标。
    loss.backward()  # 对两个 Block 执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters()))  # 汇总全部 MLP 梯度范数。
    optimizer.step()  # 应用 Adam 参数更新。
    if step % 175 == 0 or step == 699:  # 每一百七十五步保存轨迹。
        history.append({"step": step, "loss": loss.item(), "forecast_loss": forecast_loss.item(), "residual_penalty": residual_penalty.item(), "gradient_norm": gradient_norm})  # 保存损失分项和梯度。
with torch.no_grad():  # 在六个测试窗口执行预测和分解。
    normalized_test_predictions, test_debug = model(test_inputs, return_debug=True)  # 计算标准化三步预测。
    test_predictions = normalized_test_predictions * normalization_std + normalization_mean  # 还原原始订单量尺度。
print("N-BEATS训练轨迹=", history)  # 展示预测损失、残差惩罚和梯度。
print("首个测试窗口Block分解")  # 标记下方展示 backcast residual 机制。
print("input_norm=", torch.round(test_inputs[0] * 1000).tolist())  # 展示标准化十四天输入。
print("backcast1=", torch.round(test_debug["backcast_one"][0] * 1000).tolist())  # 展示第一 Block 历史解释。
print("residual1=", torch.round(test_debug["residual_one"][0] * 1000).tolist())  # 展示第二 Block 实际接收的残差。
print("forecast1/2=", torch.round(test_debug["forecast_one"][0] * 1000).tolist(), torch.round(test_debug["forecast_two"][0] * 1000).tolist())  # 展示两个未来贡献。

N-BEATS训练轨迹= [{'step': 0, 'loss': 1.297892451286316, 'forecast_loss': 1.2767943143844604, 'residual_penalty': 0.02109813317656517, 'gradient_norm': 1.6804187875402288}, {'step': 175, 'loss': 5.4779269703431055e-05, 'forecast_loss': 1.223623030455201e-06, 'residual_penalty': 5.355564644560218e-05, 'gradient_norm': 0.00021268457243706865}, {'step': 350, 'loss': 1.6524738384759985e-05, 'forecast_loss': 8.571836929149868e-08, 'residual_penalty': 1.643902032810729e-05, 'gradient_norm': 7.990671579726113e-05}, {'step': 525, 'loss': 5.6485064305888955e-06, 'forecast_loss': 2.2419150980113045e-08, 'residual_penalty': 5.626087386190193e-06, 'gradient_norm': 3.986512393456608e-05}, {'step': 699, 'loss': 2.6261891434842255e-06, 'forecast_loss': 1.2335505950034076e-08, 'residual_penalty': 2.6138536668440793e-06, 'gradient_norm': 2.2367993210907064e-05}]
首个测试窗口Block分解
input_norm= [397.0, 1275.0, 1964.0, 1702.0, 1156.0, 373.0, 112.0, 895.0, 1774.0, 2462.0, 2201.0, 1655.0, 872.0, 610.0]
backcast1= [-

## 4. 滚动测试结果与结果解读

对最后六个 origin 使用完全相同的三步目标比较 Naive 与 N-BEATS，输出逐 origin MAE 和整体 MAE。

In [4]:
nbeats_absolute_errors = torch.abs(test_predictions - test_targets_raw)  # 计算 N-BEATS 逐点绝对误差。
nbeats_mae = float(nbeats_absolute_errors.mean().item())  # 汇总六个 origin 三步 MAE。
print("origin   targets                    naive_pred/MAE                    N-BEATS_pred/MAE")  # 输出滚动对照表头。
for origin, target, baseline_prediction, baseline_errors, prediction, errors in zip(test_origins, test_targets_raw, baseline_predictions, baseline_absolute_errors, test_predictions, nbeats_absolute_errors):  # 逐 origin 比较预测。
    print(f"{dates[origin]:<8} {str([round(value, 2) for value in target.tolist()]):<26} {str([round(value, 2) for value in baseline_prediction.tolist()]):<24}/{baseline_errors.mean().item():>5.2f} {str([round(value, 2) for value in prediction.tolist()]):<24}/{errors.mean().item():>5.2f}")  # 输出当前多步预测和 MAE。
print(f"结果解读：Last-value MAE={baseline_mae:.4f}，两Block N-BEATS MAE={nbeats_mae:.4f}；模型从两周历史学习趋势与周周期。")  # 解释同数据预测收益。

origin   targets                    naive_pred/MAE                    N-BEATS_pred/MAE
day-47   [115.0, 124.25, 131.5]     [106.75, 106.75, 106.75]/16.83 [115.11, 124.5, 130.88] / 0.33
day-48   [124.25, 131.5, 128.75]    [115.0, 115.0, 115.0]   /13.17 [124.4, 131.72, 128.65] / 0.16
day-49   [131.5, 128.75, 123.0]     [124.25, 124.25, 124.25]/ 4.33 [131.64, 129.03, 122.92]/ 0.17
day-50   [128.75, 123.0, 114.75]    [131.5, 131.5, 131.5]   / 9.33 [128.99, 122.64, 114.6] / 0.25
day-51   [123.0, 114.75, 112.0]     [128.75, 128.75, 128.75]/12.17 [123.22, 114.87, 111.91]/ 0.15
day-52   [114.75, 112.0, 120.25]    [123.0, 123.0, 123.0]   / 7.33 [115.75, 113.27, 119.88]/ 0.88
结果解读：Last-value MAE=10.5278，两Block N-BEATS MAE=0.3206；模型从两周历史学习趋势与周周期。


## 5. 失败案例与修正：第二 Block 再次读取原始输入

第二 Block 在训练时学的是 residual 分布。推理若误把原始 x 再送一次，两个 forecast 会重复解释相同模式，测试误差上升。

In [5]:
with torch.no_grad():  # 在无梯度环境复现错误推理图。
    wrong_backcast_one, wrong_forecast_one = model.block_one(test_inputs)  # 正常计算第一 Block 输出。
    wrong_backcast_two, wrong_forecast_two = model.block_two(test_inputs)  # 错误地让第二 Block 重新读取原始 x。
    wrong_normalized_forecast = wrong_forecast_one + wrong_forecast_two  # 累加重复解释的两个 forecast。
    wrong_predictions = wrong_normalized_forecast * normalization_std + normalization_mean  # 还原错误预测到订单尺度。
wrong_residual_flow_mae = float(torch.abs(wrong_predictions - test_targets_raw).mean().item())  # 计算绕过 backcast residual 的测试 MAE。
correct_residual_flow_mae = nbeats_mae  # 读取正确 residual 堆叠 MAE。
print(f"错误行为：block2(input)，test MAE={wrong_residual_flow_mae:.4f}，首个预测={wrong_predictions[0].tolist()}")  # 展示重复解释造成的预测偏差。
print(f"修正行为：block2(input-backcast1)，test MAE={correct_residual_flow_mae:.4f}，首个预测={test_predictions[0].tolist()}")  # 展示正确残差路径。

错误行为：block2(input)，test MAE=2.0833，首个预测=[116.20146942138672, 126.42396545410156, 134.83331298828125]
修正行为：block2(input-backcast1)，test MAE=0.3206，首个预测=[115.1122055053711, 124.49819946289062, 130.87725830078125]


## 6. 生产边界

确定性单序列没有节假日和缺失。生产需要多商品/门店共享、已知未来特征、异常修复、时间滚动交叉验证、预测区间、层级一致性、训练统计版本、批量推理延迟和促销分布漂移监控。

In [6]:
nbeats_diagnostics = {"days": len(daily_orders), "training_windows": len(training_origins), "test_origins": len(test_origins), "input_length": input_length, "horizon": horizon, "baseline_mae": baseline_mae, "nbeats_mae": nbeats_mae, "wrong_residual_flow_mae": wrong_residual_flow_mae, "normalization_uses_training_only": True}  # 汇总数据、误差和推理图指标。
print("生产监控快照：", nbeats_diagnostics)  # 输出 N-BEATS 服务应持续观察的信号。

生产监控快照： {'days': 56, 'training_windows': 30, 'test_origins': 6, 'input_length': 14, 'horizon': 3, 'baseline_mae': 10.527777671813965, 'nbeats_mae': 0.32060369849205017, 'wrong_residual_flow_mae': 2.083332061767578, 'normalization_uses_training_only': True}


## 7. 最小回归测试

断言覆盖序列规模、真实训练、滚动预测、残差堆叠和有限输出。

In [7]:
assert len(daily_orders) >= 6 and len(training_origins) >= 6 and len(test_origins) >= 6  # 保证真实时序和滚动窗口规模充足。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证两 Block 实际 forward/backward 学习。
assert test_predictions.shape == test_targets_raw.shape and torch.isfinite(test_predictions).all()  # 保证多步预测形状正确且有限。
assert nbeats_mae < baseline_mae  # 保证相同滚动测试上 N-BEATS 优于最后值基线。
assert wrong_residual_flow_mae > correct_residual_flow_mae  # 保证绕过 backcast residual 的失败真实复现。
assert normalization_mean == training_inputs_raw.mean() and normalization_std == training_inputs_raw.std().clamp_min(1.0e-6)  # 保证归一化统计没有读取测试未来。